In [136]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Episode 01
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    3-Model Arena Comparison<br>
    <span style="color:#63b3ed;">on Your Mac</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Apple Silicon’s unified memory changes everything. Run 122B, 35B, and 0.8B
    models side-by-side to see the quality/speed tradeoff viscerally.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🍎 Apple Silicon</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 MLX</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🔌 OpenAI API</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🏆 3 Models</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~20 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Hardware Check</h2>
</div>

First — what are we working with?

In [137]:
!python3 ../../scripts/setup_check.py


┌────────────────────────────────────────────────────────────┐
│               LLM Lab — Hardware Setup Check               │
└────────────────────────────────────────────────────────────┘

  ▸ Operating System
  ────────────────────────────────────────────────────────
    OS                           Darwin 25.3.0
    Architecture                 arm64
    Python                       3.14.3

  ▸ Chip / Accelerator
  ────────────────────────────────────────────────────────
    Chip                         Apple M4 Max

  ▸ Memory (RAM)
  ────────────────────────────────────────────────────────
    Total RAM                    128.0 GiB
    Available RAM                36.8 GiB

  ▸ MLX Framework
  ────────────────────────────────────────────────────────
    ✓  mlx.core importable                       version=unknown  device=Device(gpu, 0)

  ▸ Model Recommendations
  ────────────────────────────────────────────────────────
    Detected tier: 128 GB+

    Model                       

In [138]:
# Connect to 3 MLX servers and warm up in parallel
from openai import OpenAI
import time
import concurrent.futures
from IPython.display import HTML, display

MODELS = [
    {"label": "122B", "model": "arthurcollet/Qwen3.5-122B-A10B-mlx-nvfp4", "port": 8800, "color": "#2563eb"},
    {"label": "35B", "model": "RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-nvfp4", "port": 8801, "color": "#16a34a"},
    {"label": "2B", "model": "RepublicOfKorokke/Qwen3.5-2B-mlx-lm-nvfp4", "port": 8802, "color": "#f59e0b"},
]

# Create clients
clients = {}
for m in MODELS:
    clients[m["label"]] = OpenAI(base_url=f"http://localhost:{m['port']}/v1", api_key="unused")

# Warm up all 3 in parallel
def warmup(m):
    t0 = time.time()
    try:
        clients[m["label"]].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=1,
        )
        return m["label"], time.time() - t0, True
    except Exception as e:
        return m["label"], time.time() - t0, False

print("Warming up 3 models in parallel...", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    warmup_results = list(ex.map(warmup, MODELS))

# Display status table
rows = ""
for label, elapsed, ok in warmup_results:
    status = "✓" if ok else "✗"
    color = "#16a34a" if ok else "#dc2626"
    m = next(m for m in MODELS if m["label"] == label)
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{m['port']}</td>"
        f"<td style='padding:6px 12px; color:{color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{status}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Port</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Status</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Warmup Time</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print("All models ready!")

Warming up 3 models in parallel...


Model,Port,Status,Warmup Time
122B,8800,✓,0.9s
35B,8801,✓,0.4s
2B,8802,✓,0.2s


All models ready!


In [139]:
# Helper functions for 3-model comparisons
import re
import time
import html
import markdown
import threading
import concurrent.futures
from IPython.display import HTML, display

def strip_think(text):
    """Remove <think>...</think> blocks from Qwen3.5 reasoning output."""
    cleaned = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = cleaned[:cleaned.index('<think>')]
    return cleaned.strip()

def _md(text):
    """Convert markdown text to HTML. Falls back to escaped text on error."""
    try:
        return markdown.markdown(text, extensions=["fenced_code", "tables"])
    except Exception:
        return html.escape(text)

def _render_cards(state, models_order):
    """Render 3-column HTML cards from current state."""
    cards = ""
    for m in models_order:
        s = state[m["label"]]
        text = strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = _md(text)
        if s["done"]:
            status = f"{s['tps']:.1f} tok/s"
        else:
            elapsed_str = f"{s['elapsed']:.1f}s" if s["elapsed"] > 0 else ""
            status = f"streaming... {s['tokens']} tok {elapsed_str}"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db;
                    border-left:4px solid {s['color']}; border-radius:0 8px 8px 0; padding:12px 18px;">
          <div style="color:{s['color']}; font-weight:bold; font-size:0.85em; margin-bottom:6px;">
            {m['label']} · {status}
          </div>
          <div style="color:#1f2937; line-height:1.6; word-wrap:break-word; font-size:0.9em;">
            {rendered}
          </div>
          <div style="color:#9ca3af; font-size:0.75em; margin-top:8px;">
            {s['tokens']} tokens in {s['elapsed']:.1f}s
          </div>
        </div>"""
    return f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'

def compare_models(prompt, system_prompt=None, **kwargs):
    """Fire the same prompt at all 3 models in parallel with streaming, display side-by-side cards."""
    kwargs.setdefault("max_tokens", 1024)
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})

    order = {"2B": 0, "35B": 1, "122B": 2}
    models_order = sorted(MODELS, key=lambda m: order.get(m["label"], 99))

    # Shared state for each model
    state = {}
    for m in MODELS:
        state[m["label"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "done": False, "color": m["color"]}

    handle = display(HTML(_render_cards(state, models_order)), display_id=True)

    def stream_model(m):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        t0 = time.time()
        try:
            response = clients[m["label"]].chat.completions.create(
                model=m["model"], messages=messages, stream=True, **kwargs
            )
            token_count = 0
            for chunk in response:
                if chunk.choices and chunk.choices[0].delta.content:
                    state[m["label"]]["text"] += chunk.choices[0].delta.content
                    token_count += 1
                    elapsed = time.time() - t0
                    state[m["label"]]["tokens"] = token_count
                    state[m["label"]]["elapsed"] = elapsed
                    state[m["label"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
        except Exception as e:
            state[m["label"]]["text"] = f"Error: {e}"
        finally:
            elapsed = time.time() - t0
            state[m["label"]]["elapsed"] = elapsed
            if state[m["label"]]["tokens"] > 0:
                state[m["label"]]["tps"] = state[m["label"]]["tokens"] / elapsed
            state[m["label"]]["done"] = True

    # Launch all 3 streaming threads
    threads = []
    for m in MODELS:
        t = threading.Thread(target=stream_model, args=(m,))
        t.start()
        threads.append(t)

    # Refresh display every 300ms until all done
    while not all(state[m["label"]]["done"] for m in MODELS):
        time.sleep(0.3)
        handle.update(HTML(_render_cards(state, models_order)))

    # Final render
    handle.update(HTML(_render_cards(state, models_order)))

    # Return results
    results = []
    for m in models_order:
        s = state[m["label"]]
        results.append({
            "label": m["label"], "text": strip_think(s["text"]),
            "tokens": s["tokens"], "elapsed": s["elapsed"],
            "tps": s["tps"], "color": s["color"]
        })
    return results


def show_metrics_table(results):
    """Render a comparison metrics table from compare_models results."""
    rows = ""
    for r in results:
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: First Inference — 3-Model Comparison</h2>
</div>

In [140]:
results = compare_models(
    "Explain what you are in exactly 3 sentences, as if you were describing yourself to an electrical engineer who has never heard of an LLM.",
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: Measure Performance</h2>
</div>

In [141]:
# Measure tok/s across all 3 models with a fun prompt
import psutil

results = compare_models(
    "Write a limerick about UART protocol timing violations.",
)

show_metrics_table(results)

ram = psutil.virtual_memory()
print(f"\nSystem RAM used: {(ram.total - ram.available) / 1e9:.1f} GB / {ram.total / 1e9:.0f} GB ({ram.percent}%)")

Model,Tokens,Time,tok/s
2B,1024,4.5s,227.0
35B,37,1.1s,34.2
122B,44,2.4s,18.1



System RAM used: 125.5 GB / 137 GB (91.3%)


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 4: Memory Topology</h2>
</div>

Why can Apple Silicon run all 3 models simultaneously? **Unified memory.** The CPU, GPU, and Neural Engine share the same RAM — no PCIe bottleneck, no separate VRAM pool.

In [142]:
# Show the memory topology — why Apple Silicon is special for this
import psutil
from IPython.display import HTML, display

ram = psutil.virtual_memory()
used_gb = (ram.total - ram.available) / 1e9

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
    <th style="padding:8px 12px; color:white; text-align:left;">GPU Memory</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Access</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">Discrete GPU</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">24 GB VRAM</td>
        <td style="padding:6px 12px; color:#dc2626; font-weight:bold; border-bottom:1px solid #e5e7eb;">Model must fit here</td></tr>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">This Mac</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">{ram.total/1e9:.0f} GB Unified</td>
        <td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">CPU + GPU + Neural Engine share</td></tr>
  </tbody>
</table>

<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Approx Footprint</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#2563eb; font-weight:bold; border-bottom:1px solid #e5e7eb;">122B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~65 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">35B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~20 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#f59e0b; font-weight:bold; border-bottom:1px solid #e5e7eb;">2B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~1.2 GB</td></tr>
    <tr style="background:#f0f4ff;"><td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">Total</td>
        <td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">~86.2 GB</td></tr>
  </tbody>
</table>
"""))

print(f"System RAM in use: {used_gb:.1f} GB / {ram.total/1e9:.0f} GB ({ram.percent}%)")
print(f"Available RAM: {ram.available/1e9:.1f} GB")

System RAM in use: 124.4 GB / 137 GB (90.5%)
Available RAM: 13.0 GB


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🌡️ Step 5: Temperature Experiment</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Temperature controls the randomness of token sampling. At <code>0</code> the model always picks the most likely next token (deterministic). Higher values spread probability across more tokens, making output more creative — or more chaotic.
</div>

In [146]:
# Temperature = 0.7 across all 3 models — watch the quality spread
results = compare_models(
    "In one sentence, describe how electricity flows through a wire, but make it sound like ancient mythology.",
    temperature=1.2,
)

# Try other temperatures yourself — change the value above and re-run!

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎭 Step 6: System Prompts</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> A system prompt sets the model’s persona and behavior rules. The weights don’t change — you’re just framing the input differently. Think of it as a costume for the model.
</div>

In [147]:
# Pirate persona across all 3 models — entertaining quality spread!
results = compare_models(
    "What is a neural network?",
    system_prompt="You are a pirate who somehow understands machine learning. Arr.",
    temperature=0.8,
)

In [149]:
# Bonus: 3 personas on the 122B model — streaming in parallel
# ↓ Change this and re-run to see how temperature affects each persona ↓
PERSONA_TEMP = 0.7

import time
import threading
from IPython.display import HTML, display

personas = [
    {"name": "Grumpy Firmware Engineer", "icon": "\U0001f610", "system": "You are a grumpy firmware engineer with 30 years of experience. You think everything modern is bloated and wrong. Be brief and sarcastic.", "color": "#dc2626"},
    {"name": "Excited Intern", "icon": "\U0001f929", "system": "You are an extremely enthusiastic first-year CS student who just discovered transformers. Use lots of exclamation points.", "color": "#2563eb"},
    {"name": "ML Pirate", "icon": "\U0001f3f4\u200d\u2620\ufe0f", "system": "You are a pirate who somehow understands machine learning. Arr.", "color": "#16a34a"},
]

question = "What is a neural network?"
model_122b = next(m for m in MODELS if m["label"] == "122B")

# Shared state
state = {}
for p in personas:
    state[p["name"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "done": False}

def _render_persona_cards():
    cards = ""
    for p in personas:
        s = state[p["name"]]
        text = strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = _md(text)
        if s["done"]:
            status = f"{s['tps']:.1f} tok/s"
        else:
            status = f"streaming... {s['tokens']} tok"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db; border-radius:8px;
                    padding:16px 20px; font-family:'Inter',sans-serif; word-wrap:break-word; overflow-wrap:break-word;">
          <div style="color:{p['color']}; font-weight:bold; font-size:0.9em; margin-bottom:8px;">
            {p['icon']} {p['name']} · {status}
          </div>
          <div style="color:#1f2937; line-height:1.6;">{rendered}</div>
        </div>"""
    return f"""<div style="color:#6b7280; font-size:0.85em; margin-bottom:8px;">122B model — same weights, different personas (temp={PERSONA_TEMP}):</div>
<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>"""

handle = display(HTML(_render_persona_cards()), display_id=True)

def stream_persona(p):
    t0 = time.time()
    try:
        response = clients["122B"].chat.completions.create(
            model=model_122b["model"],
            messages=[
                {"role": "system", "content": p["system"]},
                {"role": "user", "content": question},
            ],
            max_tokens=512, temperature=PERSONA_TEMP, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        token_count = 0
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                state[p["name"]]["text"] += chunk.choices[0].delta.content
                token_count += 1
                elapsed = time.time() - t0
                state[p["name"]]["tokens"] = token_count
                state[p["name"]]["elapsed"] = elapsed
                state[p["name"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
    except Exception as e:
        state[p["name"]]["text"] = f"Error: {e}"
    finally:
        elapsed = time.time() - t0
        state[p["name"]]["elapsed"] = elapsed
        if state[p["name"]]["tokens"] > 0:
            state[p["name"]]["tps"] = state[p["name"]]["tokens"] / elapsed
        state[p["name"]]["done"] = True

threads = [threading.Thread(target=stream_persona, args=(p,)) for p in personas]
for t in threads:
    t.start()

while not all(state[p["name"]]["done"] for p in personas):
    time.sleep(0.3)
    handle.update(HTML(_render_persona_cards()))

handle.update(HTML(_render_persona_cards()))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧮 Step 7: Quantization</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Quantization reduces the precision of model weights (e.g., from 32-bit floats to 4-bit integers). This shrinks the model ~8× with only a small quality loss, making it possible to fit large models in memory.
</div>

In [150]:
# What quantization actually does to numbers
import mlx.core as mx
from IPython.display import HTML, display

# Real weights are fp32 — 4 bytes each
weights_fp32 = mx.array([0.1234, -0.5678, 0.9012, -0.3456, 0.7890])

# 4-bit quantization: ~16 possible values per weight, loses precision
# Simulate by rounding to nearest 1/8
weights_4bit_approx = mx.round(weights_fp32 * 8) / 8

print("What quantization does to weights:")
print(f"  fp32:   {weights_fp32.tolist()}")
print(f"  ~4-bit: {weights_4bit_approx.tolist()}")
print()

# Size math — this is why quantization matters
params = 32_000_000_000  # 32B parameters

rows = ""
for label, mult, note in [
    ("fp32 (4 bytes/param)", 4, "doesn't fit on most machines"),
    ("bf16 (2 bytes/param)", 2, "still large"),
    ("4-bit (0.5 bytes/param)", 0.5, "fits in 64GB+ Mac"),
]:
    size = params * mult / 1e9
    rows += (
        f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{size:.0f} GB \u2014 {note}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Precision</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Size (32B params)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))

What quantization does to weights:
  fp32:   [0.1234000027179718, -0.567799985408783, 0.901199996471405, -0.3456000089645386, 0.7889999747276306]
  ~4-bit: [0.125, -0.625, 0.875, -0.375, 0.75]



Precision,Size (32B params)
fp32 (4 bytes/param),128 GB — doesn't fit on most machines
bf16 (2 bytes/param),64 GB — still large
4-bit (0.5 bytes/param),16 GB — fits in 64GB+ Mac


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💪 Step 8: Stress Test</h2>
</div>

In [151]:
# Stress test: 3 tests x 3 models = 9 combinations
import time
import concurrent.futures
from IPython.display import HTML, display

tests = [
    ("Simple", "What is 2+2? Answer with just the number.", 64),
    ("Medium", "List 5 common I2C bus errors and how to debug them.", 512),
    ("Creative", "Write a Python function that reads from a UART buffer and parses NMEA GPS sentences. Include docstring.", 1024),
]

def run_test(m, prompt, max_tokens):
    t0 = time.time()
    r = clients[m["label"]].chat.completions.create(
        model=m["model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    tokens = r.usage.completion_tokens
    return {"label": m["label"], "color": m["color"], "tokens": tokens, "elapsed": elapsed, "tps": tokens / elapsed if elapsed > 0 else 0}

order = {"2B": 0, "35B": 1, "122B": 2}

for test_name, prompt, max_tok in tests:
    print(f"Running: {test_name} (max_tokens={max_tok})...", flush=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(run_test, m, prompt, max_tok): m["label"] for m in MODELS}
        test_results = []
        for f in concurrent.futures.as_completed(futures):
            test_results.append(f.result())
    test_results.sort(key=lambda r: order.get(r["label"], 99))

    rows = ""
    for r in test_results:
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <div style="color:#1e3a5f; font-weight:bold; margin:8px 0 4px 0;">{test_name}</div>
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:4px 0 12px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))
    print(f"  ✓ {test_name} complete\n", flush=True)

print("All stress tests done.")

Running: Simple (max_tokens=64)...


Model,Tokens,Time,tok/s
2B,2,0.3s,8.0
35B,2,0.5s,4.0
122B,4,1.1s,3.7


  ✓ Simple complete

Running: Medium (max_tokens=512)...


Model,Tokens,Time,tok/s
2B,512,4.0s,126.7
35B,512,8.9s,57.7
122B,512,12.1s,42.5


  ✓ Medium complete

Running: Creative (max_tokens=1024)...


Model,Tokens,Time,tok/s
2B,1024,8.0s,127.8
35B,740,13.6s,54.5
122B,1024,23.2s,44.1


  ✓ Creative complete

All stress tests done.


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🗂️ Step 9: KV Cache — Why Second Turns Are Faster</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> The KV (key-value) cache stores attention matrices from prior tokens so the model doesn’t recompute them on every turn. This speeds up multi-turn conversations but grows with context length — long conversations eat more RAM.
</div>

In [152]:
# KV cache demo: measure time-to-first-token (TTFT) for cold vs cached
# TTFT isolates prompt processing time — the phase KV cache speeds up
import time
from IPython.display import HTML, display

def kv_cache_test(m):
    """Measure TTFT for cold (new conversation) vs cached (follow-up turn)."""
    # Turn 1 — cold: no prior context, must process prompt from scratch
    messages_1 = [{"role": "user", "content": "I'm going to ask you 3 questions about embedded systems. Ready?"}]
    t0 = time.time()
    stream_1 = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages_1, max_tokens=50, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    ttft_cold = None
    reply_1 = ""
    for chunk in stream_1:
        if ttft_cold is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_cold = time.time() - t0
        if chunk.choices and chunk.choices[0].delta.content:
            reply_1 += chunk.choices[0].delta.content
    if ttft_cold is None:
        ttft_cold = time.time() - t0

    # Turn 2 — cached: server reuses KV cache from turn 1
    messages_2 = messages_1 + [
        {"role": "assistant", "content": reply_1},
        {"role": "user", "content": "Question 1: What's the difference between I2C and SPI?"},
    ]
    t1 = time.time()
    stream_2 = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages_2, max_tokens=50, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    ttft_cached = None
    for chunk in stream_2:
        if ttft_cached is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_cached = time.time() - t1
            break  # got what we need
    if ttft_cached is None:
        ttft_cached = time.time() - t1

    speedup = ttft_cold / ttft_cached if ttft_cached > 0 else 0
    return {"label": m["label"], "color": m["color"], "cold": ttft_cold, "cached": ttft_cached, "speedup": speedup}

print("Measuring time-to-first-token (TTFT)...", flush=True)
results = []
for m in MODELS:
    print(f"  Testing {m['label']}...", flush=True)
    results.append(kv_cache_test(m))

rows = ""
for r in results:
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cold']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cached']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['speedup']:.1f}x</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT cold</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT cached</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speedup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  TTFT = time-to-first-token — measures prompt processing, not generation.
  Cached turn reuses KV matrices from turn 1, so only new tokens need processing.
</div>
"""))
print("Done.")

Measuring time-to-first-token (TTFT)...
  Testing 122B...
  Testing 35B...
  Testing 2B...


Model,TTFT cold,TTFT cached,Speedup
122B,996 ms,321 ms,3.1x
35B,433 ms,285 ms,1.5x
2B,196 ms,175 ms,1.1x


Done.


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Unified memory** lets Apple Silicon run models that don’t fit on discrete GPUs
- **MLX** serves models locally via an OpenAI-compatible API
- **4-bit quantization** shrinks models 8× with minimal quality loss
- **Temperature** controls randomness: 0 = deterministic, 1.5 = chaotic
- **System prompts** steer behavior without changing model weights
- **tok/s** is bounded by memory bandwidth, not compute
- **KV cache** speeds up multi-turn conversations by reusing prior computations
- **Running multiple models simultaneously** reveals the quality/speed tradeoff viscerally
- **Model size vs quality:** 0.8B is instant but shallow, 35B hits the sweet spot, 122B is best but slowest
- Everything here ran locally — no data left your machine

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What’s Next</h2>
</div>

- **Episode 02:** Structured output — making LLMs return JSON you can actually parse
- **Episode 03:** Tool use — letting models call functions and interact with external systems
- **Episode 04:** RAG — grounding responses in your own documents
- **Episode 05:** Fine-tuning — teaching a model your domain